# Documentación completa: `forward.py` + `utils.py`
### Modelo LBL (Line-By-Line) de transmitancia y atenuación atmosférica
**Photonix Team · 2026**

Este notebook explica de forma exhaustiva:
- La **física** detrás del modelo (fórmulas con LaTeX)
- La **lógica de código** (qué hace cada función, en qué orden)
- Los **tipos y dimensiones** de cada variable
- Cómo se **relacionan** `forward.py` y `utils.py`


## 1. Arquitectura general

```
instance_lsf.py  (script de usuario)
    │
    ├─ importa forward.py ──────► utils.py
    │       │                        │
    │       │  run_simulation()      │  _prepare_forward_variants()
    │       │  Species (dataclass)   │  default_species()
    │       │  Isotopologue          │  _isotopologue_bank()
    │       │                        │  load_Q_vals()
    │       │                        │  read_hitran_par_minimal()
    │       │                        │  transmittance_for_gas_tile()
    │       │                        │   └─ voigt_profile()
    │       │                        │  bin_average()
    │       │                        │  apply_lsf_nu()
    │       │                        │   └─ _gauss_kernel_nu()
    │       │                        │  _dedup_by_nu()
    │       │                        │  convert_atm(), convert_vmr()
    │       │                        │  downsample(), txt_to_npy(), ...
    └─ llama a run_simulation() ───────────────────────────────────
```

> **Regla clave:** `forward.py` contiene la lógica de orquestación (`run_simulation`) y los dataclasses. `utils.py` contiene toda la física y los helpers numéricos.


## 2. Constantes físicas

Definidas en **ambos archivos** (idénticas) para evitar importaciones circulares:

| Variable | Valor | Unidad | Significado |
|---|---|---|---|
| `TREF` | 296.0 | K | Temperatura de referencia HITRAN |
| `K_B_erg` | 1.3806×10⁻¹⁶ | erg/K | Constante de Boltzmann en CGS |
| `C_M_S` | 2.9979×10⁸ | m/s | Velocidad de la luz |
| `C_CM_S` | 2.9979×10¹⁰ | cm/s | Velocidad de la luz en CGS |
| `C2` | 1.43880285 | cm·K | Segunda constante de radiación ($hc/k_B$) |
| `N_A` | 6.0221×10²³ | mol⁻¹ | Número de Avogadro |
| `N_L` | 2.4794×10¹⁹ | cm⁻³ | Número de Loschmidt (densidad a STP) |

**¿Por qué CGS?** HITRAN reporta intensidades de línea en cm⁻¹/(molécula·cm⁻²), por lo que toda la física espectral usa CGS. Solo al final se convierte a µm para graficar.


## 3. Estructuras de datos: `Isotopologue` y `Species`

Definidas en `forward.py`. Son los **objetos centrales** que viajan por todo el sistema.

### 3.1 `Isotopologue`
```python
@dataclass
class Isotopologue:
    iso:   str           # ID HITRAN del isotopólogo, ej. '1', '2', 'A'
    qfile: str           # ruta al archivo Q(T), ej. 'tips/H2O/q1.txt'
    Wg:    float         # masa molar del isotopólogo [g/mol]
    Pmol:  float | None  # presión parcial [atm] — None hereda la del padre
```
Es un contenedor **liviano**, solo para isotopólogos secundarios.

### 3.2 `Species`
```python
@dataclass
class Species:
    name:   str                    # 'H2O', 'CO2', etc.
    mol:    int                    # número de molécula HITRAN (H2O=1, CO2=2, ...)
    iso:    str                    # isotopólogo principal, ej. '1'
    qfile:  str                    # ruta al Q(T) del isotopólogo principal
    Wg:     float                  # masa molar [g/mol]
    Pmol:   float                  # presión parcial [atm]
    extra_isotopologues: List[Isotopologue]  # isotopólogos adicionales
    Qref:   float  # calculado en run_simulation: Q(TREF)
    QT:     float  # calculado en run_simulation: Q(T_simulación)
    idx_all: np.ndarray | None     # máscara booleana sobre H (tabla HITRAN)
```

Un `Species` puede tener **múltiples isotopólogos** (ej. CO₂ tiene 12). Cuando `use_all_isotopologues=True`, cada isotopólogo se trata como una variante independiente.


## 4. Banco de isotopólogos y `default_species()` (en `utils.py`)

### `_isotopologue_bank(base='tips/')` → `Dict[str, Dict]`
Diccionario hardcodeado con todos los isotopólogos de HITRAN para 7 gases.
Cada entrada tiene:
- `mol`: número de molécula HITRAN
- `Pmol`: concentración atmosférica estándar (fracción molar)
- `variants`: lista de `Isotopologue` con sus archivos Q(T)

### `default_species()` → `List[Species]`
Construye la lista estándar de 7 gases: **H₂O, CO₂, O₃, N₂O, CO, CH₄, O₂**.
Para cada gas:
1. Toma el primer isotopólogo como **principal** (`main`)
2. El resto van a `extra_isotopologues`
3. Crea un objeto `Species` listo para usar

```python
# En instance_lsf.py:
sp = default_species()   # Lista[Species] con 7 elementos
for s in sp:
    if s.name == 'H2O': s.Pmol = 0.00775 * Ptotal  # sobreescribir VMR
```

**Tipos de salida:** `List[Species]`, longitud 7 (uno por gas).


## 5. Expansión de variantes: `_prepare_forward_variants()` (en `utils.py`)

```python
forward_variants, variant_to_parent = _prepare_forward_variants(species, use_all_isotopologues)
```

### ¿Para qué sirve?
El código necesita calcular la transmitancia de **cada isotopólogo por separado** (porque cada uno tiene su propio Q(T) y masa molar). Esta función 'explota' la lista de Species en una lista plana de variantes.

### Ejemplo (CO₂ tiene 12 isotopólogos):
```
Entrada: species = [Species('H2O', iso='1', ...), Species('CO2', iso='1', ...)]

use_all=False:  forward_variants = [H2O_iso1, CO2_iso1]          len=2
                variant_to_parent = [0, 1]

use_all=True:   forward_variants = [H2O_iso1..7, CO2_iso1..12]   len=19
                variant_to_parent = [0,0,0,0,0,0,0, 1,1,1,1,...]
```

### Retorna:
- `forward_variants`: `List[Species]` — cada elemento es un isotopólogo individual
- `variant_to_parent`: `List[int]` — índice del gas padre en `species` para cada variante

### Uso posterior en `run_simulation`:
```python
# Para acumular transmitancias por gas padre:
for variant_idx, parent_idx in enumerate(variant_to_parent):
    T_species_ext[parent_idx] *= T_ext_each_variants[variant_idx, :]
```
Es decir, la transmitancia total de H₂O = **producto** de todos sus isotopólogos.


## 6. Lectura del archivo HITRAN: `read_hitran_par_minimal()` (en `utils.py`)

### Formato HITRAN 2004 (160 caracteres por línea)
```
Cols  1- 2  mol       Número de molécula (int)
Col   3     iso       ID de isotopólogo  (str)
Cols  4-15  nu0       Posición de línea  [cm⁻¹]  (float64)
Cols 16-25  Sref      Intensidad a Tref  [cm⁻¹/(mol·cm⁻²)] (float64)
Cols 26-35  A         Coeficiente Einstein A [s⁻¹] (float64)
Cols 36-40  g_air     Anchura Lorentz aire  [cm⁻¹/atm] (float64)
Cols 41-45  g_self    Anchura Lorentz self  [cm⁻¹/atm] (float64)
Cols 46-55  Elow      Energía estado bajo  [cm⁻¹] (float64)
Cols 56-59  n_air     Exponente temperatura (float64)
Cols 60-67  shift     Presión-shift [cm⁻¹/atm] (float64)
```
El código **solo lee las primeras 10 columnas** (anchuras fijas definidas en `widths`).

### Retorna: `Dict[str, np.ndarray]`
```
H = {
  'mol':    np.int32[N_lineas],   # ej. [1,1,1,...,2,2,...]
  'iso':    np.str_[N_lineas],    # ej. ['1','1','2',...]
  'nu0':    np.float64[N_lineas], # posición espectral [cm⁻¹]
  'Sref':   np.float64[N_lineas], # intensidad a 296 K
  'A':      np.float64[N_lineas], # coef. Einstein
  'g_air':  np.float64[N_lineas],
  'g_self': np.float64[N_lineas],
  'Elow':   np.float64[N_lineas],
  'n_air':  np.float64[N_lineas],
  'shift':  np.float64[N_lineas],
}
```
`N_lineas` puede ser del orden de **millones** para el archivo `ALL.par`.

### ¿Por qué `<U10` para `iso`?
Algunos isotopólogos de CO₂ tienen ID de letra: `'A'`, `'B'`. NumPy usa strings Unicode.


## 7. Función de partición Q(T): `load_Q_vals()` (en `utils.py`)

### ¿Qué es Q(T)?
La **función de partición total** Q(T) cuantifica cómo se distribuye la población de moléculas entre sus niveles cuánticos a temperatura T:

$$Q(T) = \sum_i g_i \, e^{-E_i / k_B T}$$

Es necesaria para escalar las intensidades de línea de la temperatura de referencia (296 K) a la temperatura real de simulación.

### `load_Q_vals(qfile, Tref=296, T)` → `(Qref: float, QT: float)`
1. Lee el archivo `tips/H2O/q1.txt` (dos columnas: T, Q)
2. Interpola linealmente en Tref → `Qref = Q(296 K)`
3. Interpola linealmente en T → `QT = Q(T_simulación)`

Los archivos Q(T) vienen de la base de datos **TIPS** (Total Internal Partition Sums).

```python
# Llamada en run_simulation:
for var in forward_variants:
    var.Qref, var.QT = load_Q_vals(var.qfile, TREF, temp_K)
```

**Tipos:** `Qref` y `QT` son `float` escalares. Se asignan a los campos del `Species`.


## 8. Tiling espectral (en `run_simulation`, `forward.py`)

### ¿Por qué tiles?
Calcular el perfil de Voigt para **millones de líneas** sobre una grilla de millones de puntos es una operación O(N_lineas × N_puntos). Para hacerlo manejable en memoria y tiempo, el espectro se divide en **tiles** (ventanas solapadas).

```python
edges_tiles = np.arange(nu_min, nu_max + 1e-9, tileW)  # bordes de tiles [cm⁻¹]
```

### Parámetros clave:
| Parámetro | Típico | Significado |
|---|---|---|
| `tileW` | 20.0 cm⁻¹ | Ancho del tile (parte que se guarda) |
| `guard` | 25.0 cm⁻¹ | Margen extra a cada lado (para las alas de Lorentz) |
| `dnu` | 0.01–0.1 cm⁻¹ | Resolución espectral |

### Por cada tile [a, b] → [a−guard, b+guard]:
```
nu_ext: array float64, tamaño ≈ (tileW + 2*guard)/dnu ≈ 7000 puntos (con dnu=0.01)
```

### ¿Por qué el margen `guard`?
Las alas de Lorentz de una línea en `a-5 cm⁻¹` contribuyen dentro del tile [a,b]. Sin `guard`, se perderían esas contribuciones y habría discontinuidades en los bordes.

### Acumulación:
```python
nu_all_parts     → después: np.concatenate → nu_all        float64[M_total]
T_prod_all_parts → después: np.concatenate → T_prod        float64[M_total]
T_each_acc       → después: np.concatenate → T_each        List[float64[M_total]]
```
donde M_total ≈ (nu_max - nu_min) / dnu


## 9. Física LBL: `transmittance_for_gas_tile()` (en `utils.py`)

Esta es la **función más importante** del modelo. Calcula la transmitancia espectral de un gas para un tile dado.

### Firma:
```python
transmittance_for_gas_tile(
    nu_vec:     np.ndarray,       # grilla espectral extendida [cm⁻¹], shape (M,)
    H:          Dict[str, ndarray],# tabla HITRAN completa
    sp:         Species,          # especie + isotopólogo a calcular
    Tgas:       float,            # temperatura [K]
    pres:       float,            # presión total [atm]
    Lm:         float,            # longitud de camino [m]
    mask_lines: np.ndarray,       # bool[N_lineas], True = líneas de este tile
) -> np.ndarray                   # transmitancia, shape (M,), valores en (0,1]
```

### Paso 1 — Extraer líneas del tile:
```python
nu0, Sref, g_air, g_self, Elow, n_air, shift = H[...][mask_lines]
# Cada uno: float64[K] donde K = número de líneas en el tile extendido
```

### Paso 2 — Presiones parciales:
```python
Pself = sp.Pmol          # presión parcial del gas [atm]
Pair  = pres - Pself     # presión del resto del aire [atm]
```

### Paso 3 — Presión-shift de la posición de línea:
$$\nu'_0 = \nu_0 + \delta \cdot P_{air}$$

donde $\delta$ (`shift`) es el coeficiente de desplazamiento por presión [cm⁻¹/atm].

### Paso 4 — Escala de intensidad con temperatura:
La intensidad de línea a temperatura T se calcula desde la intensidad de referencia $S(T_{ref})$:

$$S(T) = S(T_{ref}) \cdot \frac{Q(T_{ref})}{Q(T)} \cdot \frac{e^{-C_2 E'' / T}}{e^{-C_2 E'' / T_{ref}}} \cdot \frac{1 - e^{-C_2 \nu_0 / T}}{1 - e^{-C_2 \nu_0 / T_{ref}}}$$

donde:
- $Q(T)$: función de partición a temperatura T
- $E''$: energía del estado fundamental (`Elow`) [cm⁻¹]
- $C_2 = hc/k_B = 1.4388$ cm·K
- El último cociente corrige la **emisión estimulada**

```python
S_T = (Sref * (sp.Qref / sp.QT) *
       np.exp(-C2*Elow/Tgas) / np.exp(-C2*Elow/TREF) *
       (1.0 - np.exp(-C2*nu0/Tgas)) / (1.0 - np.exp(-C2*nu0/TREF)))
# S_T: float64[K]
```

### Paso 5 — Intensidad total de la línea (con densidad de columna):
$$I_{line} = S(T) \cdot \frac{T_{ref}}{T} \cdot N_L \cdot P_{self} \cdot L_{cm}$$

- $N_L = 2.479 \times 10^{19}$ cm⁻³ (número de Loschmidt): convierte presión parcial a densidad de moléculas
- $L_{cm} = L_m \times 100$: longitud de camino en cm
- El factor $T_{ref}/T$ corrige la densidad numérica real a temperatura T

```python
line_intensity = S_T * (TREF/Tgas) * N_L * Pself * Lcm  # float64[K]
```

### Paso 6 — Anchura Doppler (Gaussiana):
$$\alpha = \frac{\nu_0}{c} \sqrt{\frac{2 N_A k_B T \ln 2}{W_g}}$$

donde $W_g$ es la masa molar del isotopólogo [g/mol].

```python
alpha = nu0 / C_CM_S * np.sqrt(2*N_A * K_B_erg * Tgas * np.log(2) / sp.Wg)
# alpha: float64[K] (semi-anchura a mitad de altura, en cm⁻¹)
```

### Paso 7 — Anchura Lorentz (presión):
$$\gamma = \left(\frac{T_{ref}}{T}\right)^{n_{air}} \left( g_{air} \cdot P_{air} + g_{self} \cdot P_{self} \right)$$

```python
gamma = ((TREF/Tgas)**n_air) * (g_air * Pair + g_self * Pself)
# gamma: float64[K] (semi-anchura Lorentz, en cm⁻¹)
```


## 10. Perfil de Voigt: `voigt_profile()` (en `utils.py`)

### ¿Qué es el perfil de Voigt?
La forma de línea real es la **convolución** de un perfil Gaussiano (ensanchamiento Doppler) y un perfil Lorentziano (ensanchamiento por presión):

$$f_V(\nu) = \frac{\sqrt{\ln 2}}{\sqrt{\pi}\, \alpha} \, \text{Re}\left[w(z)\right]$$

donde $w(z)$ es la **función de Faddeeva** y:

$$z = \sqrt{\ln 2} \frac{(\nu - \nu'_0)}{\alpha} + i\, \sqrt{\ln 2} \frac{\gamma}{\alpha}$$

### Implementación:
```python
def voigt_profile(nu, nu0_shifted, alpha, gamma):
    # nu:          float64[M]  ← grilla espectral del tile
    # nu0_shifted: float64[K]  ← posiciones de línea (K líneas)
    # alpha:       float64[K]  ← anchura Doppler
    # gamma:       float64[K]  ← anchura Lorentz

    s2 = sqrt(ln2)
    x = s2 * (nu[:,None] - nu0_shifted[None,:]) / alpha[None,:]  # shape (M, K)
    y = s2 * (gamma[None,:] / alpha[None,:])                      # shape (M, K)
    z = x + 1j*y
    w = wofz(z)        # función de Faddeeva de scipy, shape (M, K)
    fV = s2/sqrt(pi)/alpha[None,:] * Re(w)  # shape (M, K)
    return fV          # float64[M, K]
```

### Broadcasting:
El truco clave es usar `nu[:, None]` y `nu0[None, :]` para crear la **matriz de diferencias** de forma eficiente sin bucles. El resultado es una matriz `(M, K)` donde cada columna es el perfil de una línea evaluado en todos los puntos de la grilla.

### Profundidad óptica (paso 8):
```python
tau = fV @ line_intensity   # (M,K) @ (K,) → float64[M]
```
Esto es la **suma** sobre todas las K líneas del tile:
$$\tau(\nu) = \sum_k f_V^{(k)}(\nu) \cdot I_{line}^{(k)}$$

### Transmitancia:
$$T(\nu) = e^{-\tau(\nu)}$$
```python
return np.exp(-tau)   # float64[M], valores en (0.0, 1.0]
```

> **Nota de memoria:** La matriz `(M, K)` puede ser grande. Con M=7000 y K=500 líneas, son 3.5M floats = ~28 MB. Para K muy grande puede necesitar chunking, pero actualmente el código lo hace en una sola operación.


## 11. Acumulación de transmitancias en `run_simulation` (`forward.py`)

Después de calcular la transmitancia de cada variante en el tile:

```python
T_ext_each_variants  # float64[n_variantes, M_ext]
```

### Transmitancia combinada (producto):
$$T_{total}(\nu) = \prod_{k=1}^{N_{variantes}} T_k(\nu)$$

```python
T_ext_prod = np.prod(T_ext_each_variants, axis=0)  # float64[M_ext]
```

### Transmitancia por gas (agrupando isotopólogos del mismo padre):
```python
T_species_ext  # float64[n_gases, M_ext], inicializado a 1.0

for variant_idx, parent_idx in enumerate(variant_to_parent):
    T_species_ext[parent_idx] *= T_ext_each_variants[variant_idx, :]
# ↑ equivale a: T_gas = producto de todos sus isotopólogos
```

### Suma de transmitancias (para referencia):
```python
T_ext_sum = np.sum(T_species_ext, axis=0)  # float64[M_ext]
```
(Suma aritmética de transmitancias por gas — útil como diagnóstico, no es la transmitancia física real).

### Recorte del tile:
```python
keep = (nu_ext >= a) & (nu_ext <= b)  # bool[M_ext]
# Solo se guarda la parte central, sin los márgenes 'guard'
```


## 12. Función de Dispersión de Línea (LSF): `apply_lsf_nu()` (en `utils.py`)

### ¿Qué es el LSF?
Modela la **respuesta instrumental**: cualquier espectrómetro real tiene una resolución finita. El LSF convierte el espectro ideal de alta resolución en lo que mediría el instrumento.

### Kernel Gaussiano: `_gauss_kernel_nu(FWHM_cm1, dnu)`

$$k[n] = e^{-\frac{1}{2}\left(\frac{n \cdot d\nu}{\sigma}\right)^2}, \quad \sigma = \frac{FWHM}{2\sqrt{2\ln 2}}$$

```python
k = _gauss_kernel_nu(FWHM_cm1=0.05, dnu=0.01)
# k: float64[2*half+1], normalizado a suma=1.0
# half = int(4*sigma/dnu) → longitud del kernel ≈ 8*sigma/dnu + 1
```

### Dos dominios de convolución (`domain`):

**`domain='tau'`** (recomendado físicamente):
$$\tau_{conv}(\nu) = (\tau * k)(\nu), \quad T_{out}(\nu) = e^{-\tau_{conv}(\nu)}$$

```python
tau = -log(clip(T, 1e-300, 1.0))
tau_conv = convolve(tau, k, mode='same')
T_out = exp(-tau_conv)
```

**`domain='T'`** (convolución directa en transmitancia):
$$T_{out}(\nu) = (T * k)(\nu)$$

La diferencia: convolucionar en $\tau$ preserva mejor los picos de absorción profundos.

### Firma:
```python
apply_lsf_nu(
    T_each_species_nu: List[float64[M]],  # transmitancias por gas en dominio nu
    dnu: float,
    kind: str,      # 'gaussian' (único implementado)
    W_cm1: float,   # FWHM del LSF en cm⁻¹
    domain: str,    # 'tau' o 'T'
) -> (List[float64[M]], float64[M], Dict)
      T_each_conv    T_total_conv   meta
```

### `_dedup_by_nu()` — promediado de duplicados
Antes del LSF, si los tiles tienen puntos `nu` duplicados en sus fronteras, se promedian para tener una grilla uniforme:
```python
nu_uniq, arrays_averaged = _dedup_by_nu(nu_all, [T_prod, T_sum] + T_each)
```


## 13. Conversión a longitud de onda y binning: `bin_average()` (en `utils.py`)

### Conversión de números de onda a µm:
$$\lambda \, [\mu m] = \frac{10^4}{\nu \, [cm^{-1}]}$$

```python
lambda_um = 1e4 / nu_all          # float64[M_total]
ord_idx   = np.argsort(lambda_um) # ordena de menor a mayor wavelength
lambda_sorted = lambda_um[ord_idx]
```

**Importante:** La conversión invierte el orden (nu grande → lambda pequeño), por eso se necesita `argsort`.

### Binning: `bin_average(x_sorted, y_sorted, edges)` → `(centers, yb)`
Agrupa los puntos de alta resolución en bins de ancho `delta_um`:

```python
edges = np.arange(lam_min, lam_max + 1e-9, delta_um)  # Bordes de bins [µm]
# edges: float64[N_bins+1]

centers, T_samp = bin_average(lambda_sorted, T_lambda, edges)
# centers: float64[N_bins] = medios de cada bin
# T_samp:  float64[N_bins] = promedio de T dentro del bin
```

**Lógica interna de `bin_average`:**
- Usa `np.searchsorted` para encontrar los índices de los bordes (O(log N) por bin)
- Si el bin tiene puntos: `mean(y[l:r])`
- Si el bin está vacío (resolución muy baja): `np.interp` al centro del bin

### Dimensiones finales:
```
lambda_centers: float64[N_bins]         N_bins = (lam_max-lam_min)/delta_um
T_prod_samp:    float64[N_bins]
T_each_samp:    List[float64[N_bins]]   len = n_gases
```
Con `nu_min=666.67, nu_max=2500, delta_um=0.1`: N_bins ≈ ~100 (solo banda 4-15 µm).


## 14. Cálculo de atenuación: `att=True` (en `run_simulation`, `forward.py`)

### De transmitancia a atenuación en dB/m:

$$A \, [dB/m] = -\frac{10}{L} \log_{10}(T)$$

donde $L$ es la longitud de camino en metros.

```python
invL = 1.0 / L_m
A_dbm = -(10.0 * invL) * np.log10(np.clip(T_samp, 1e-300, 1.0))
# float64[N_bins]
```

**¿Por qué `clip`?** Para evitar `log10(0) = -inf`. Se limita a 1e-300 como valor mínimo.

### Atenuación total (suma de contribuciones por gas):
$$A_{total} = \sum_{gases} A_{gas}$$

Esto es válido porque:
$$-\log_{10}\left(\prod_k T_k\right) = \sum_k -\log_{10}(T_k)$$

```python
A_dbm_lam_sum = np.sum(np.stack(A_dbm_lam_each, axis=0), axis=0)  # float64[N_bins]
```


## 15. Funciones utilitarias de soporte (en `utils.py`)

### `convert_atm(P, u)` → `float`
Convierte presión al atmosférico [atm]:
- `u='mbar'`: divide por 1013.25
- `u='gcms2'`: divide por 1013.25×10³ (unidades CGS de presión)

### `convert_vmr(ppm)` → `float`
Convierte ppm a fracción molar: `ppm × 10⁻⁶`. Raramente usado directamente.

### `txt_to_npy(txt_path, out_path)`
Lee un archivo de texto con 2 columnas (longitud de onda, transmitancia) y lo guarda como `.npy` con forma `(2, N)`.

### `downsample(simulated_dir, gt_dir, output_dir)`
Remuestrea datos de ground truth (alta resolución) a la grilla de la simulación, usando integración trapezoidal + `cumsum`:
```
gt_dir[0]:  lambda_src float64[N_src]    grilla fuente (alta res)
gt_dir[1]:  tau_src    float64[N_src]    valores fuente
simulated_dir[0]: lam_tgt float64[N_tgt] grilla objetivo (baja res)
output → .npy con shape (2, N_tgt)
```

### `count_lines_by_index(parfile, index)` → `int`
Cuenta cuántas líneas HITRAN tiene la molécula con índice `index`. Útil para diagnóstico.

### `plot_npy`, `plot_histogram`
Funciones de visualización rápida para exploración de datos. No se usan en el flujo principal.


## 16. Flujo completo paso a paso

```
instance_lsf.py
│
│  1. convert_atm(1013, 'mbar')          → Ptotal [atm] (float)
│  2. default_species()                   → species List[Species] (len=7)
│  3. overwrite Pmol for each gas
│
└──► run_simulation(species, ...)
      │
      │  [A] _prepare_forward_variants(species, use_all=True)
      │      → forward_variants List[Species]  (len=N_variantes)
      │      → variant_to_parent List[int]
      │
      │  [B] load_Q_vals(var.qfile, TREF, temp_K)  ← por cada variante
      │      → var.Qref, var.QT  (float, float)
      │
      │  [C] read_hitran_par_minimal('pars/ALL.par')
      │      → H  Dict[str, float64[N_lineas]]
      │
      │  [D] Para cada variante: H['mol']==mol & H['iso']==iso
      │      → var.idx_all  bool[N_lineas]
      │
      │  [E] LOOP TILES  (edges_tiles = arange(nu_min, nu_max, tileW))
      │      │
      │      │  [E.1] nu_ext = arange(a-guard, b+guard, dnu)   float64[~7000]
      │      │
      │      │  [E.2] Por cada variante k:
      │      │         mask_tile = idx_all & (nu0 in [a_ext, b_ext])
      │      │         T_ext_each_variants[k] =
      │      │           transmittance_for_gas_tile(nu_ext, H, var, ...)
      │      │             → voigt_profile(nu_ext, nu0s, alpha, gamma)
      │      │                 → tau = fV @ line_intensity
      │      │                 → exp(-tau)   float64[~7000]
      │      │
      │      │  [E.3] T_ext_prod = prod(T_ext_each_variants, axis=0)
      │      │  [E.4] T_species_ext[parent] *= T_ext_each_variants[k]
      │      │  [E.5] Recortar a [a, b] y acumular en listas
      │      │
      │  [F] Concatenar todas las partes:
      │      nu_all  float64[M]   T_prod  float64[M]   T_each List[float64[M]]
      │
      │  [G] (Opcional) apply_lsf_nu(T_each, dnu, W_cm1=0.05, domain='tau')
      │      → T_each, T_prod actualizados
      │
      │  [H] lambda_um = 1e4/nu_all
      │      argsort → lambda_sorted, T_sorted
      │
      │  [I] bin_average(..., edges)  ← delta_um=0.1 µm
      │      → lambda_centers float64[N_bins]
      │      → T_prod_samp    float64[N_bins]
      │      → T_each_samp    List[float64[N_bins]]
      │
      │  [J] att=True: A_dbm = -(10/L)*log10(T_samp)
      │
      │  [K] make_plots=True: guardar .png/.pdf en outdir/
      │  [L] save_csv=True:   guardar .csv en outdir/
      │  [M] transmission_npy_name: guardar .npy [2,N_bins]
      │
      └── return dict  (lambda_centers, T_prod_samp, T_each_samp,
                         A_dbm_lam_sum, A_dbm_lam_each, species, lsf, ...)
```


## 17. Tabla resumen de funciones

| Función | Archivo | Input principal | Output | Propósito |
|---|---|---|---|---|
| `default_species()` | utils | — | `List[Species]` len=7 | Crea los 7 gases estándar |
| `_isotopologue_bank()` | utils | — | `Dict` | Catálogo de isotopólogos |
| `_prepare_forward_variants()` | utils | `List[Species]` | `List[Species]`, `List[int]` | Explota isotopólogos |
| `load_Q_vals()` | utils | path, Tref, T | `(float, float)` | Lee Q(Tref), Q(T) |
| `read_hitran_par_minimal()` | utils | path .par | `Dict[str, ndarray]` | Parsea HITRAN |
| `voigt_profile()` | utils | nu[M], nu0[K], α[K], γ[K] | `float64[M,K]` | Perfil de línea |
| `transmittance_for_gas_tile()` | utils | nu[M], H, Species, ... | `float64[M]` | T de 1 gas en 1 tile |
| `bin_average()` | utils | x[N], y[N], edges[B+1] | `(float64[B], float64[B])` | Remuestrea a bins |
| `apply_lsf_nu()` | utils | `List[float64[M]]`, dnu, ... | `List[float64[M]], float64[M], Dict` | Aplica LSF |
| `_gauss_kernel_nu()` | utils | FWHM, dnu | `float64[2H+1]` | Kernel Gaussiano |
| `_dedup_by_nu()` | utils | nu[M], List[arr] | mismas dims sin duplic. | Limpia duplicados |
| `convert_atm()` | utils | P, unidad | `float` | Convierte presión |
| `downsample()` | utils | sim, gt arrays | guarda .npy | Resamplea GT a grilla sim |
| `run_simulation()` | forward | `List[Species]`, params... | `Dict` | Orquesta todo |


## 18. Dimensiones típicas de arrays

Con los parámetros de `instance_lsf.py` (`nu_min=666.67`, `nu_max=2500`, `dnu=0.1`, `tileW=20`, `guard=25`, `delta_um=0.1`):

| Variable | Shape | Descripción |
|---|---|---|
| `H['nu0']` | `(N_lineas,)` | ~millones de líneas en ALL.par |
| `nu_ext` (por tile) | `(~700,)` | tileW+2×guard / dnu = 70/0.1 |
| `T_ext_each_variants` | `(N_var, ~700)` | por tile, todas las variantes |
| `nu_all` (después de concat) | `(~18330,)` | (2500-666.67)/dnu ≈ 18333 |
| `lambda_sorted` | `(~18330,)` | mismo tamaño, ordenado en µm |
| `lambda_centers` | `(~90,)` | (15−4)/0.1 ≈ 110 bins (4–15 µm) |
| `T_each_samp` | `List[(~90,)]` | 1 array por gas |
| `A_dbm_lam_each` | `List[(~90,)]` | atenuación por gas |

> **Nota:** Con `species_to_use=['H2O','CO2']` solo se simulan 2 gases, reduciendo el tiempo de cómputo significativamente.


## 19. Resumen físico de la ecuación completa

El modelo calcula, para un camino atmosférico de longitud $L$ a temperatura $T$ y presión $P$:

$$\boxed{T(\lambda) = \exp\left(-\sum_{gases} \sum_{lineas\,k} I_{line}^{(k)} \cdot f_V^{(k)}(\nu(\lambda))\right)}$$

donde:
- $I_{line}^{(k)} = S(T)^{(k)} \cdot \frac{T_{ref}}{T} \cdot N_L \cdot P_{self} \cdot L_{cm}$ — densidad de columna ponderada por intensidad
- $f_V^{(k)}$ — perfil de Voigt (convolución Gaussiano×Lorentziano)
- La transmitancia total es el **producto** de todos los gases: $T_{total} = \prod_{gases} T_{gas}$
- La atenuación en dB/m: $A = -\frac{10}{L} \log_{10} T$

Opcionalmente, el resultado se convuelve con el LSF instrumental:
$$\tau_{medido}(\nu) = \tau_{ideal}(\nu) * k_{LSF}(\nu)$$
